# JetRacer — Follow an Object at a Fixed Distance

This notebook makes the **Waveshare JetRacer ROS AI Kit** (JetRacer built on NVIDIA Jetson
Nano/Orin Nano) detect a chosen object with the onboard camera and drive toward/away from it
so that it keeps a **constant distance**, while steering to keep the object centered.

It is written to be run cell-by-cell inside Jupyter Lab on the robot itself (the same way the
stock Waveshare/JetRacer notebooks — `basic_motion.ipynb`, `interactive_regression.ipynb`,
etc. — are run).

## What this notebook covers

1. Hardware/software assumptions and safety notes
2. Camera initialization (CSI camera via `jetcam`)
3. Motor/steering control initialization (via `jetracer.nvidia_racecar`)
4. A lightweight on-device object detector (MobileNet-SSD via OpenCV's DNN module)
5. Picking the "target" object out of all detections (closest / most confident)
6. Estimating relative distance from bounding-box size (monocular proxy for depth)
7. Two independent PID controllers — one for **steering** (centering), one for
   **throttle** (distance-keeping)
8. A calibration procedure so "fixed distance" means something physical (e.g. 50 cm)
9. The live control loop, with a video widget and on-screen sliders for tuning
10. Safety watchdogs (lost-target stop, throttle clamps, emergency stop button)
11. Clean shutdown

> ⚠️ **Read the Safety Notes section before running the motion cells.** Always test with the
> car on a stand (wheels off the ground) first, and keep a hand near the emergency-stop
> button / power switch.

---


## 0. Hardware & Software Assumptions

- **Board**: Jetson Nano (or Orin Nano) as shipped in the JetRacer ROS AI Kit Professional
  Version.
- **Camera**: CSI ribbon camera (default in the kit). If you use a USB camera instead, swap
  `CSICamera` for `USBCamera` where noted in Section 2.
- **Drive stack**: `jetracer` Python package (`NvidiaRacecar` class) driving a PCA9685 PWM
  controller → ESC (throttle) + steering servo. This is the same library used by the stock
  NVIDIA-AI-IOT JetRacer notebooks that Waveshare's image is based on.
- **Camera helper**: `jetcam` Python package (`CSICamera` / `USBCamera`), also standard on the
  JetRacer/JetBot images.
- **OpenCV**: must be built with the `dnn` module (true for the JetPack OpenCV that ships on
  the Jetson image). We use OpenCV's DNN module rather than `jetson-inference` so this
  notebook has no extra compiled dependencies beyond what the stock image already has.

If any import in Section 1 fails, the message will tell you which package is missing and
this notebook explains how to work around it.

### Safety notes

- The throttle values used by `NvidiaRacecar` are tiny floats (roughly -1.0 to 1.0). This
  notebook enforces a **hard throttle ceiling** (`MAX_THROTTLE`) far below 1.0 — raise it
  gradually and only once you've verified steering/stopping behave as expected.
- The control loop includes a **watchdog**: if the target object is not seen for
  `LOST_TARGET_TIMEOUT` seconds, the car stops automatically.
- There is a manual **STOP** widget button that immediately zeroes throttle and steering.
- Always keep the car on blocks (wheels free-spinning) the first time you run Section 9.


In [ ]:
# ---------------------------------------------------------------------------
# 1. Imports & sanity checks
# ---------------------------------------------------------------------------
import time
import traceback
import threading

import numpy as np
import cv2

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError as e:
    raise ImportError(
        "ipywidgets is required for the live view / sliders. "
        "Install with: pip3 install ipywidgets"
    ) from e

try:
    from jetracer.nvidia_racecar import NvidiaRacecar
except ImportError as e:
    raise ImportError(
        "Could not import jetracer. This notebook expects the JetRacer software stack "
        "(https://github.com/NVIDIA-AI-IOT/jetracer) which ships preinstalled on the "
        "Waveshare JetRacer image. If it's missing: "
        "git clone https://github.com/NVIDIA-AI-IOT/jetracer && cd jetracer && "
        "python3 setup.py install --user"
    ) from e

try:
    from jetcam.csi_camera import CSICamera
    # from jetcam.usb_camera import USBCamera  # uncomment if you use a USB webcam instead
except ImportError as e:
    raise ImportError(
        "Could not import jetcam. This ships preinstalled on the Waveshare JetRacer image. "
        "If missing: git clone https://github.com/NVIDIA-AI-IOT/jetcam && cd jetcam && "
        "python3 setup.py install --user"
    ) from e

print("cv2 version:", cv2.__version__)
print("cv2 has dnn module:", hasattr(cv2, "dnn"))


## 2. Camera Initialization

`CSICamera` grabs frames from the ribbon-cable camera in a background thread and stores the
latest one in `camera.value` (as a BGR numpy array, HxWx3, uint8) — exactly like in the stock
JetRacer/JetBot notebooks.

- `width`/`height`: capture resolution. **300×300** is used here (rather than the 224×224 in
  the stock JetRacer notebooks) because the detector's input is also 300×300 — capturing at
  that size avoids the extra blur/aliasing you get from upscaling a smaller frame, which
  measurably helps small or partially-visible people get detected. This has a minor cost in
  loop speed; drop back to 224 if you need more FPS and can stand closer/more centered.
- `capture_fps`: keep at 30 (sensor default) — we throttle the *processing* rate separately
  in the control loop, not the sensor.

If your kit uses a USB camera instead of the CSI ribbon camera, replace this cell's
`CSICamera(...)` with `USBCamera(width=300, height=300, capture_device=0)` (import at the top
already has the commented-out line for this).


In [ ]:
# ---------------------------------------------------------------------------
# 2. Camera Initialization
# ---------------------------------------------------------------------------
CAMERA_WIDTH = 300
CAMERA_HEIGHT = 300
CAMERA_FPS = 30

camera = CSICamera(width=CAMERA_WIDTH, height=CAMERA_HEIGHT, capture_fps=CAMERA_FPS)
camera.running = True  # starts the background capture thread

time.sleep(1.0)  # let the sensor warm up / first frames arrive
test_frame = camera.value
print("Camera OK, frame shape:", None if test_frame is None else test_frame.shape)


## 3. Motor / Steering Initialization

`NvidiaRacecar` talks to the PCA9685 PWM driver that controls:

- `car.steering` — float, roughly **-1.0 (full left) to +1.0 (full right)**, 0.0 = straight.
- `car.throttle` — float, roughly **-1.0 (full reverse) to +1.0 (full forward)**, 0.0 = stop.

### Calibration constants

Every physical car is a little different (servo center point isn't perfectly 0, one motor
channel might be reversed, etc.). The stock JetRacer notebooks calibrate these interactively;
we expose the same two knobs here:

- `steering_gain` / `steering_offset`: adjust until the front wheels point straight when you
  command `car.steering = 0`, and turn proportionally (not too twitchy, not too sluggish) as
  the value moves toward ±1.
- `MAX_THROTTLE`: hard ceiling this notebook will never exceed, regardless of what the PID
  controller computes. **Start low (e.g. 0.15–0.20) and only raise it once you trust the
  behavior.**

If you already calibrated these in the kit's own setup notebook, copy the same values here.


In [ ]:
# ---------------------------------------------------------------------------
# 3. Motor / Steering Initialization
# ---------------------------------------------------------------------------
car = NvidiaRacecar()

# --- Calibration (tune these for your specific car) ---
car.steering_gain = -0.65      # sign/magnitude depends on your servo wiring; flip sign if
                                # the car turns the wrong way
car.steering_offset = 0.0      # small trim so 0.0 == wheels dead straight

MAX_THROTTLE = 0.18            # hard safety ceiling (raise gradually after testing)
MIN_MOVE_THROTTLE = 0.10       # below this the motor may not have enough torque to move at all
THROTTLE_SIGN = 1.0            # set to -1.0 in Section 6c if forward/reverse turn out swapped
REVERSE_STEERING_SIGN = -1.0   # flip steering when reversing (see note in Section 9). If your
                                # car turns the "wrong way" only while reversing after this fix,
                                # set this to 1.0 instead.

# Always start stopped
car.throttle = 0.0
car.steering = 0.0
print("Car initialized. steering_gain=%.2f steering_offset=%.2f MAX_THROTTLE=%.2f"
      % (car.steering_gain, car.steering_offset, MAX_THROTTLE))


## 4. Object Detector (SSD via OpenCV DNN)

Two pretrained SSD models are supported, controlled by `USE_COCO_MODEL` below:

- **COCO model (default, recommended)** — `ssd_mobilenet_v2_coco` (TensorFlow), trained on
  Microsoft COCO, which has roughly **10x more person examples in far more varied poses,
  clothing, lighting, and partial-body crops** than the alternative below. This is the fix for
  "it doesn't detect me easily" in most cases — a broader, more diverse training set makes the
  detector far more robust to a person standing close to the camera, at an angle, partially
  out of frame, backlit, etc. 80 classes total.
- **VOC model (fallback)** — the original `MobileNet-SSD` (Caffe) used in earlier versions of
  this notebook, trained on PASCAL VOC (only 20 classes, smaller/older dataset). Kept as an
  automatic fallback in case the COCO model can't be downloaded (e.g. no internet on the
  Jetson, or its host is unreachable from your network).

Both run through OpenCV's `dnn` module — no extra compiled dependencies beyond what the stock
Jetson image already has.

**Why not `jetson-inference`?** It's faster, but requires a separate compiled install.
`detect_objects()` in the next cells is the only thing you'd need to swap out if you'd rather
use `jetson-inference`'s `detectNet` — everything downstream (target selection, PID, control
loop) stays the same.


In [ ]:
# ---------------------------------------------------------------------------
# 4a. Download the model files (run once)
# ---------------------------------------------------------------------------
import os
import tarfile
import shutil
import urllib.request

USE_COCO_MODEL = True   # <-- set False to force the smaller/older VOC (20-class) model

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

# --- COCO model (TensorFlow) file locations ---
COCO_PB_PATH = os.path.join(MODEL_DIR, "ssd_mobilenet_v2_coco_2018_03_29", "frozen_inference_graph.pb")
COCO_PBTXT_PATH = os.path.join(MODEL_DIR, "ssd_mobilenet_v2_coco_2018_03_29.pbtxt")
COCO_TARBALL_URL = "http://download.tensorflow.org/models/object_detection/ssd_mobilenet_v2_coco_2018_03_29.tar.gz"
COCO_PBTXT_URL = (
    "https://raw.githubusercontent.com/opencv/opencv_extra/master/testdata/dnn/"
    "ssd_mobilenet_v2_coco_2018_03_29.pbtxt"
)
MIN_COCO_PB_BYTES = 50_000_000     # real file is ~67 MB
MIN_COCO_PBTXT_BYTES = 50_000      # real file is ~113 KB

# --- VOC model (Caffe) file locations, used as a fallback ---
PROTOTXT_PATH = os.path.join(MODEL_DIR, "MobileNetSSD_deploy.prototxt")
CAFFEMODEL_PATH = os.path.join(MODEL_DIR, "MobileNetSSD_deploy.caffemodel")
PROTOTXT_URL = "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/deploy.prototxt"
CAFFEMODEL_URL = (
    "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/"
    "mobilenet_iter_73000.caffemodel"
)
MIN_PROTOTXT_BYTES = 10_000        # real file is ~44 KB
MIN_CAFFEMODEL_BYTES = 20_000_000  # real file is ~23 MB


def _download_file(url, path, min_bytes, label):
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        print("Already present and looks complete:", path, "(%d bytes)" % os.path.getsize(path))
        return
    print("Downloading %s -> %s" % (label, path))
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    urllib.request.urlretrieve(url, path)
    size = os.path.getsize(path)
    if size < min_bytes:
        os.remove(path)
        raise RuntimeError(
            "Downloaded %s is only %d bytes (expected at least %d) - truncated or an error "
            "page, not the real file. Removed it; re-run this cell." % (label, size, min_bytes)
        )
    print("Done:", path, size, "bytes")


def _download_and_extract_coco():
    if os.path.exists(COCO_PB_PATH) and os.path.getsize(COCO_PB_PATH) >= MIN_COCO_PB_BYTES:
        print("Already present:", COCO_PB_PATH)
    else:
        tar_path = os.path.join(MODEL_DIR, "ssd_mobilenet_v2_coco_2018_03_29.tar.gz")
        print("Downloading COCO model tarball (~67MB, may take a while) ...")
        urllib.request.urlretrieve(COCO_TARBALL_URL, tar_path)
        print("Extracting ...")
        with tarfile.open(tar_path) as tf:
            tf.extractall(MODEL_DIR)
        os.remove(tar_path)
        if not os.path.exists(COCO_PB_PATH) or os.path.getsize(COCO_PB_PATH) < MIN_COCO_PB_BYTES:
            raise RuntimeError(
                "Extraction did not produce a valid frozen_inference_graph.pb at %s" % COCO_PB_PATH
            )
        print("Done:", COCO_PB_PATH, os.path.getsize(COCO_PB_PATH), "bytes")
    _download_file(COCO_PBTXT_URL, COCO_PBTXT_PATH, MIN_COCO_PBTXT_BYTES, "COCO pbtxt config")


ACTIVE_MODEL = None  # set to "coco" or "voc" below, used by Section 4b

if USE_COCO_MODEL:
    try:
        _download_and_extract_coco()
        ACTIVE_MODEL = "coco"
    except Exception as e:
        print("COCO model download/extraction failed:", e)
        print("Falling back to the smaller VOC model. If your Jetson has no internet access, "
              "or download.tensorflow.org is unreachable from your network, this fallback is "
              "expected -- the VOC model below still works, just less robustly on people.")
        print("(To retry the COCO model later: re-run this cell after checking connectivity, "
              "or manually download %s and %s into the '%s' folder.)"
              % (COCO_TARBALL_URL, COCO_PBTXT_URL, MODEL_DIR))

if ACTIVE_MODEL is None:
    _download_file(PROTOTXT_URL, PROTOTXT_PATH, MIN_PROTOTXT_BYTES, "VOC prototxt")
    _download_file(CAFFEMODEL_URL, CAFFEMODEL_PATH, MIN_CAFFEMODEL_BYTES, "VOC caffemodel")
    ACTIVE_MODEL = "voc"

print("Active model:", ACTIVE_MODEL)


In [ ]:
# ---------------------------------------------------------------------------
# 4b. Load the network
# ---------------------------------------------------------------------------
# Standard 90-category COCO id -> name map used by TF Object Detection API SSD models
# (note the intentional gaps in ids -- that's correct, not a typo).
COCO_LABELS = {
    1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane", 6: "bus", 7: "train",
    8: "truck", 9: "boat", 10: "traffic light", 11: "fire hydrant", 13: "stop sign",
    14: "parking meter", 15: "bench", 16: "bird", 17: "cat", 18: "dog", 19: "horse",
    20: "sheep", 21: "cow", 22: "elephant", 23: "bear", 24: "zebra", 25: "giraffe",
    27: "backpack", 28: "umbrella", 31: "handbag", 32: "tie", 33: "suitcase", 34: "frisbee",
    35: "skis", 36: "snowboard", 37: "sports ball", 38: "kite", 39: "baseball bat",
    40: "baseball glove", 41: "skateboard", 42: "surfboard", 43: "tennis racket",
    44: "bottle", 46: "wine glass", 47: "cup", 48: "fork", 49: "knife", 50: "spoon",
    51: "bowl", 52: "banana", 53: "apple", 54: "sandwich", 55: "orange", 56: "broccoli",
    57: "carrot", 58: "hot dog", 59: "pizza", 60: "donut", 61: "cake", 62: "chair",
    63: "couch", 64: "potted plant", 65: "bed", 67: "dining table", 70: "toilet", 72: "tv",
    73: "laptop", 74: "mouse", 75: "remote", 76: "keyboard", 77: "cell phone",
    78: "microwave", 79: "oven", 80: "toaster", 81: "sink", 82: "refrigerator", 84: "book",
    85: "clock", 86: "vase", 87: "scissors", 88: "teddy bear", 89: "hair drier",
    90: "toothbrush",
}

VOC_LABELS = [
    "background", "aeroplane", "bicycle", "bird", "boat",
    "bottle", "bus", "car", "cat", "chair",
    "cow", "diningtable", "dog", "horse", "motorbike",
    "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

if ACTIVE_MODEL == "coco":
    for _p, _min in [(COCO_PB_PATH, MIN_COCO_PB_BYTES), (COCO_PBTXT_PATH, MIN_COCO_PBTXT_BYTES)]:
        if not os.path.exists(_p) or os.path.getsize(_p) < _min:
            raise FileNotFoundError("%s is missing/incomplete -- re-run Section 4a." % _p)
    net = cv2.dnn.readNetFromTensorflow(COCO_PB_PATH, COCO_PBTXT_PATH)
    DETECTOR_INPUT_SIZE = (300, 300)
    _BLOB_SCALEFACTOR = 1.0
    _BLOB_MEAN = (0, 0, 0)
    _BLOB_SWAP_RB = True   # this graph was exported expecting RGB input

    def _label_for_class_id(class_id):
        return COCO_LABELS.get(class_id)

    print("Loaded COCO model (80 classes).")
else:
    for _p, _min in [(PROTOTXT_PATH, MIN_PROTOTXT_BYTES), (CAFFEMODEL_PATH, MIN_CAFFEMODEL_BYTES)]:
        if not os.path.exists(_p) or os.path.getsize(_p) < _min:
            raise FileNotFoundError("%s is missing/incomplete -- re-run Section 4a." % _p)
    net = cv2.dnn.readNetFromCaffe(PROTOTXT_PATH, CAFFEMODEL_PATH)
    DETECTOR_INPUT_SIZE = (300, 300)
    _BLOB_SCALEFACTOR = 0.007843
    _BLOB_MEAN = (127.5, 127.5, 127.5)
    _BLOB_SWAP_RB = False

    def _label_for_class_id(class_id):
        if 0 <= class_id < len(VOC_LABELS):
            return VOC_LABELS[class_id]
        return None

    print("Loaded VOC model (20 classes).")

# Use the GPU backend if available (falls back to CPU automatically on failure)
try:
    net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
    net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
    print("Using CUDA backend for inference.")
except Exception:
    net.setPreferableBackend(cv2.dnn.DNN_BACKEND_DEFAULT)
    net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
    print("CUDA backend unavailable, using CPU.")

CONFIDENCE_THRESHOLD = 0.35   # lower than the usual 0.5 -- easier to trigger on a partially
                              # visible / close-up / awkwardly-posed person. Also live-tunable
                              # via the slider in Section 8.

def detect_objects(bgr_frame, confidence_threshold=None):
    """Run the active SSD model on a BGR frame.

    Returns a list of dicts: {'label': str, 'confidence': float, 'box': (x1, y1, x2, y2)}
    where the box coordinates are in pixels in the ORIGINAL frame's coordinate system.
    """
    if confidence_threshold is None:
        confidence_threshold = CONFIDENCE_THRESHOLD

    h, w = bgr_frame.shape[:2]
    blob = cv2.dnn.blobFromImage(
        bgr_frame, scalefactor=_BLOB_SCALEFACTOR, size=DETECTOR_INPUT_SIZE,
        mean=_BLOB_MEAN, swapRB=_BLOB_SWAP_RB, crop=False
    )
    net.setInput(blob)
    raw = net.forward()  # shape: (1, 1, N, 7)

    detections = []
    for i in range(raw.shape[2]):
        confidence = float(raw[0, 0, i, 2])
        if confidence < confidence_threshold:
            continue
        class_id = int(raw[0, 0, i, 1])
        label = _label_for_class_id(class_id)
        if label is None or label == "background":
            continue
        box = raw[0, 0, i, 3:7] * np.array([w, h, w, h])
        x1, y1, x2, y2 = box.astype(int)
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w - 1, x2), min(h - 1, y2)
        detections.append({"label": label, "confidence": confidence, "box": (x1, y1, x2, y2)})
    return detections

print("Detector ready (%s model)." % ACTIVE_MODEL)


### 4c. Detection diagnostic — see what the detector is *actually* seeing

If detection still feels unreliable after switching models, run this cell while standing in
frame. It ignores `CONFIDENCE_THRESHOLD` and shows every candidate the network considered,
sorted by score. This tells you whether the detector is "almost" seeing you (e.g. person at
0.28 when the threshold is 0.35 -- just lower the threshold) versus not seeing you at all
(e.g. nothing above 0.05 -- more likely a framing/lighting/distance problem, see the tips
below the cell).


In [ ]:
# ---------------------------------------------------------------------------
# 4c. Detection diagnostic (ignores CONFIDENCE_THRESHOLD)
# ---------------------------------------------------------------------------
_diag_frame = camera.value
_diag_detections = detect_objects(_diag_frame, confidence_threshold=0.05)
_diag_detections.sort(key=lambda d: d["confidence"], reverse=True)

if not _diag_detections:
    print("No candidates at all above 0.05 confidence for ANY class.")
    print("Try: better/more even lighting, stepping back so more of your body is in frame,")
    print("or reducing backlighting (e.g. bright windows directly behind you).")
else:
    print("Top candidates this frame:")
    for d in _diag_detections[:8]:
        print("  %-16s confidence=%.2f  box=%s" % (d["label"], d["confidence"], d["box"]))
    person_hits = [d for d in _diag_detections if d["label"] == "person"]
    if person_hits:
        print()
        print("Best 'person' confidence seen: %.2f (current CONFIDENCE_THRESHOLD=%.2f)"
              % (person_hits[0]["confidence"], CONFIDENCE_THRESHOLD))
        if person_hits[0]["confidence"] < CONFIDENCE_THRESHOLD:
            print("--> This is below your threshold, which is why it's not being tracked.")
            print("    Lower CONFIDENCE_THRESHOLD above, or use the live slider in Section 8.")
    else:
        print()
        print("No 'person' candidate at all -- likely a framing/lighting/distance issue rather")
        print("than a threshold issue.")


## 5. Choosing the Target Object

Set `TARGET_LABEL` to whatever you want the car to follow. Must be one of the class names the
active model knows about: the 80 COCO names (`COCO_LABELS` values in Section 4b — e.g.
`"person"`, `"bottle"`, `"chair"`, `"cat"`, `"dog"`, `"cup"`, `"backpack"`, ...) if
`USE_COCO_MODEL = True`, or the 20 VOC names (`VOC_LABELS` in Section 4b) if it fell back to
that model. Run the Section 4c diagnostic cell to see exactly which labels the detector is
currently outputting.

If several instances of that class are visible, we assume the **largest bounding box is the
closest one** and treat it as "the" target — a reasonable heuristic for a single-object
following task and simple to reason about.


In [ ]:
# ---------------------------------------------------------------------------
# 5. Target selection
# ---------------------------------------------------------------------------
TARGET_LABEL = "person"   # <-- change this to the class you want to follow

def box_area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)

def get_target(detections, label=TARGET_LABEL):
    """Return the single best detection matching `label`, or None if not present.

    'Best' = largest bounding box among matches (closest object), tie-broken by confidence.
    """
    candidates = [d for d in detections if d["label"] == label]
    if not candidates:
        return None
    candidates.sort(key=lambda d: (box_area(d["box"]), d["confidence"]), reverse=True)
    return candidates[0]


## 6. Estimating Relative Distance from Bounding-Box Size

A single RGB camera can't measure true depth directly, but for a fixed real-world object size
(e.g. "a person," or a specific box/ball you'll always use as the target), the **bounding-box
height in pixels is inversely proportional to distance** — bigger box = closer object. That
gives us everything we need for *relative* distance control without any extra hardware:

- Define `TARGET_BOX_HEIGHT_PX`: the bounding-box height (in pixels, at the current camera
  resolution) that the object has **when it's at the distance you want the car to hold**.
- The throttle controller then just drives the *current* box height toward
  `TARGET_BOX_HEIGHT_PX`:
  - current box smaller than target → object is farther than desired → drive forward
  - current box larger than target → object is closer than desired → reverse / stop
  - equal → hold position

### Calibrating `TARGET_BOX_HEIGHT_PX`

1. Place the object you'll follow at the exact distance you want the car to maintain (e.g.
   50 cm from the camera).
2. Run the calibration cell below. It grabs a frame, runs the detector, and prints the
   detected box height for `TARGET_LABEL`.
3. Copy that printed value into `TARGET_BOX_HEIGHT_PX` in the next cell.

This is the same idea used by simple monocular "follow me" demos; it's approximate (assumes
a roughly constant real-world object size) but works well in practice for keeping a fixed
following distance.


In [ ]:
# ---------------------------------------------------------------------------
# 6a. Calibration helper — run this with the object placed at your desired distance
# ---------------------------------------------------------------------------
_frame = camera.value
_dets = detect_objects(_frame)
_target = get_target(_dets)

if _target is None:
    print("No '%s' detected. Make sure it's clearly visible, then re-run this cell." % TARGET_LABEL)
else:
    x1, y1, x2, y2 = _target["box"]
    print("Detected '%s' with confidence %.2f" % (_target["label"], _target["confidence"]))
    print("Bounding box height (px):", y2 - y1)
    print("--> Copy this value into TARGET_BOX_HEIGHT_PX in the next cell.")


In [ ]:
# ---------------------------------------------------------------------------
# 6b. Distance-control constants
# ---------------------------------------------------------------------------
TARGET_BOX_HEIGHT_PX = 150  # <-- ALWAYS replace with the value from the calibration cell above.
                             # This default is just a rough placeholder for a person standing
                             # a few steps from the camera at 224x224 resolution -- it will NOT
                             # match your setup. Skipping calibration is the #1 cause of the
                             # car only ever backing away (it happens whenever this value is
                             # smaller than the object's real apparent size at any distance
                             # you'd actually stand at).

# Tolerance band (in px) where we consider the car "at the right distance" and just idle
# the throttle instead of hunting back and forth around the setpoint.
DISTANCE_DEADBAND_PX = 6

def distance_error(target_detection):
    """Positive => object is farther than desired (box smaller than target) => drive forward.
    Negative => object is closer than desired (box bigger than target) => reverse/back off."""
    x1, y1, x2, y2 = target_detection["box"]
    current_height = y2 - y1
    return TARGET_BOX_HEIGHT_PX - current_height


### 6c. Throttle-direction sanity check (run this once, car on a stand)

Before trusting the control loop, confirm that a **positive** `car.throttle` actually spins the
wheels **forward**. Some ESCs are wired/calibrated so positive PWM means reverse, or need a
brief zero-throttle pulse before they'll accept a direction change — if that's the case here,
the follow logic above will look "backwards" no matter how well it's tuned.

Run the cell below with the car on a stand (wheels off the ground). It applies a small forward
pulse, then a small reverse pulse, 1 second apart, and stops. Watch which way the wheels
actually spin for each.


In [ ]:
# ---------------------------------------------------------------------------
# 6c. Throttle-direction sanity check
# ---------------------------------------------------------------------------
TEST_THROTTLE = 0.16

print("Applying POSITIVE throttle (%.2f) for 1s -- wheels should spin FORWARD..." % TEST_THROTTLE)
car.throttle = TEST_THROTTLE
time.sleep(1.0)
car.throttle = 0.0
time.sleep(0.5)

print("Applying NEGATIVE throttle (%.2f) for 1s -- wheels should spin BACKWARD..." % -TEST_THROTTLE)
car.throttle = -TEST_THROTTLE
time.sleep(1.0)
car.throttle = 0.0

print("Done. If the directions were swapped (positive = backward), set THROTTLE_SIGN = -1.0")
print("back in Section 3's calibration cell, re-run it, then re-run Section 9 onward.")


## 7. PID Controllers

Two independent, standard PID loops:

- **Steering PID** — input: horizontal offset of the target's bounding-box center from the
  frame's horizontal center (in pixels, normalized to roughly [-1, 1]). Output: `car.steering`.
  Goal: keep the object centered left/right.
- **Throttle PID** — input: `distance_error` from Section 6 (in pixels, normalized). Output:
  `car.throttle`. Goal: keep the object's apparent size (≈ distance) constant.

Start with the provided gains (they're intentionally conservative) and tune `Kp` first, then
add a little `Kd` to damp oscillation, and only add `Ki` if there's a persistent steady-state
offset (usually not needed for this task).


In [ ]:
# ---------------------------------------------------------------------------
# 7. PID controller
# ---------------------------------------------------------------------------
class PID:
    def __init__(self, kp, ki, kd, output_limits=(-1.0, 1.0)):
        self.kp, self.ki, self.kd = kp, ki, kd
        self.output_min, self.output_max = output_limits
        self._integral = 0.0
        self._prev_error = 0.0
        self._prev_time = None

    def reset(self):
        self._integral = 0.0
        self._prev_error = 0.0
        self._prev_time = None

    def step(self, error):
        now = time.time()
        dt = (now - self._prev_time) if self._prev_time is not None else 0.0
        self._prev_time = now

        self._integral += error * dt if dt > 0 else 0.0
        derivative = ((error - self._prev_error) / dt) if dt > 0 else 0.0
        self._prev_error = error

        output = self.kp * error + self.ki * self._integral + self.kd * derivative
        return float(np.clip(output, self.output_min, self.output_max))


# --- Steering PID: input is normalized horizontal offset in [-1, 1] ---
steering_pid = PID(kp=0.9, ki=0.0, kd=0.25, output_limits=(-1.0, 1.0))

# --- Throttle PID: input is normalized distance error in roughly [-1, 1] ---
throttle_pid = PID(kp=0.8, ki=0.05, kd=0.1, output_limits=(-1.0, 1.0))

print("PID controllers ready.")
print("steering_pid gains:", steering_pid.kp, steering_pid.ki, steering_pid.kd)
print("throttle_pid gains:", throttle_pid.kp, throttle_pid.ki, throttle_pid.kd)


## 8. Live View & Controls (widgets)

- An `Image` widget shows the camera feed with the detected bounding box drawn on it, so you
  can watch what the detector sees while tuning.
- Sliders let you live-tune the PID gains and `TARGET_BOX_HEIGHT_PX` without re-running cells.
- **Start** / **Stop** buttons control the control loop thread. Stop always zeroes the motors
  immediately, and doubles as the emergency stop.


In [ ]:
# ---------------------------------------------------------------------------
# 8. Widgets
# ---------------------------------------------------------------------------
image_widget = widgets.Image(format='jpeg', width=CAMERA_WIDTH * 2, height=CAMERA_HEIGHT * 2)

status_label = widgets.Label(value="Status: stopped")

steering_kp_slider = widgets.FloatSlider(value=steering_pid.kp, min=0.0, max=2.0, step=0.05, description='Steer Kp')
steering_kd_slider = widgets.FloatSlider(value=steering_pid.kd, min=0.0, max=1.0, step=0.05, description='Steer Kd')

throttle_kp_slider = widgets.FloatSlider(value=throttle_pid.kp, min=0.0, max=2.0, step=0.05, description='Dist Kp')
throttle_ki_slider = widgets.FloatSlider(value=throttle_pid.ki, min=0.0, max=0.5, step=0.01, description='Dist Ki')

target_height_slider = widgets.IntSlider(value=TARGET_BOX_HEIGHT_PX, min=10, max=CAMERA_HEIGHT,
                                          step=1, description='Target H (px)')

max_throttle_slider = widgets.FloatSlider(value=MAX_THROTTLE, min=0.0, max=0.5, step=0.01,
                                           description='Max thr.')

confidence_slider = widgets.FloatSlider(value=CONFIDENCE_THRESHOLD, min=0.05, max=0.9, step=0.05,
                                         description='Min conf.')

start_button = widgets.Button(description="START", button_style='success')
stop_button = widgets.Button(description="STOP", button_style='danger')

controls_box = widgets.VBox([
    widgets.HBox([start_button, stop_button]),
    status_label,
    steering_kp_slider, steering_kd_slider,
    throttle_kp_slider, throttle_ki_slider,
    target_height_slider,
    max_throttle_slider,
    confidence_slider,
])

display(widgets.HBox([image_widget, controls_box]))


## 9. The Control Loop

Each iteration:

1. Grab the latest camera frame.
2. Run the detector, pick the target (Section 4/5).
3. **If no target found**: increment a "lost" timer. If it exceeds `LOST_TARGET_TIMEOUT`,
   stop the car (`throttle = steering = 0`) and hold until the object reappears.
4. **If target found**: reset the lost timer, compute
   - horizontal offset error → steering PID → `car.steering`
   - distance error (Section 6) → throttle PID → `car.throttle`, clamped to `MAX_THROTTLE`
     (and to `MIN_MOVE_THROTTLE`/0 inside the deadband so the car doesn't creep).
5. Draw the detection box + center crosshair on the frame and push it to `image_widget`.
6. Sleep briefly to cap the loop rate (`CONTROL_LOOP_HZ`) — this keeps CPU/GPU usage
   reasonable and the PID's `dt` well-behaved.

### Why steering needs to flip when reversing

For a front-steered (Ackermann) vehicle, the same steering angle produces the **opposite**
yaw direction depending on whether the car is moving forward or backward — the same reason
turning a real car's steering wheel right while backing up curves the car left, not right.
Our steering command is computed purely from where the target sits in the camera image, so
it has no idea it needs to flip. The control loop below detects when `throttle_cmd` is
negative (car reversing) and multiplies the steering output by `REVERSE_STEERING_SIGN` in
that case only, so the target still ends up centered whether the car is approaching or backing
away from it. If, after this fix, reverse turning still looks wrong, flip
`REVERSE_STEERING_SIGN` to `1.0` in Section 3.

The loop runs in a background thread so the notebook stays responsive (you can move the
sliders while it's running); `Stop` sets `running_flag[0] = False`, which the thread checks
every iteration.


In [ ]:
# ---------------------------------------------------------------------------
# 9. Control loop
# ---------------------------------------------------------------------------
CONTROL_LOOP_HZ = 15
LOST_TARGET_TIMEOUT = 1.0  # seconds without a detection before we force-stop

running_flag = [False]
_loop_thread = [None]

def _draw_overlay(frame, target_detection, target_box_height_px):
    frame = frame.copy()
    h, w = frame.shape[:2]
    cv2.line(frame, (w // 2, 0), (w // 2, h), (255, 255, 255), 1)
    if target_detection is not None:
        x1, y1, x2, y2 = target_detection["box"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cx = (x1 + x2) // 2
        cy = (y1 + y2) // 2
        cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
        cv2.putText(frame, "%s %.2f" % (target_detection['label'], target_detection['confidence']),
                    (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        cv2.putText(frame, "target h=%d cur h=%d" % (target_box_height_px, y2 - y1),
                    (5, h - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 0), 1)
    return frame

def _control_loop():
    period = 1.0 / CONTROL_LOOP_HZ
    lost_since = None
    steering_pid.reset()
    throttle_pid.reset()

    while running_flag[0]:
        loop_start = time.time()
        try:
            # live-apply slider values
            steering_pid.kp = steering_kp_slider.value
            steering_pid.kd = steering_kd_slider.value
            throttle_pid.kp = throttle_kp_slider.value
            throttle_pid.ki = throttle_ki_slider.value
            target_h_px = target_height_slider.value
            max_throttle = max_throttle_slider.value
            confidence_threshold = confidence_slider.value

            frame = camera.value
            if frame is None:
                time.sleep(period)
                continue

            detections = detect_objects(frame, confidence_threshold=confidence_threshold)
            target = get_target(detections, TARGET_LABEL)

            if target is None:
                if lost_since is None:
                    lost_since = time.time()
                elapsed_lost = time.time() - lost_since
                status_label.value = "Status: target lost (%.1fs)" % elapsed_lost
                if elapsed_lost >= LOST_TARGET_TIMEOUT:
                    car.throttle = 0.0
                    car.steering = 0.0
            else:
                lost_since = None
                h, w = frame.shape[:2]
                x1, y1, x2, y2 = target["box"]
                cx = (x1 + x2) / 2.0

                # --- Steering: normalized horizontal offset in [-1, 1] ---
                horiz_offset_norm = (cx - w / 2.0) / (w / 2.0)
                raw_steering_cmd = steering_pid.step(horiz_offset_norm)

                # --- Throttle: normalized distance error in roughly [-1, 1] ---
                current_h = y2 - y1
                err_px = target_h_px - current_h
                if abs(err_px) <= DISTANCE_DEADBAND_PX:
                    throttle_cmd = 0.0
                    throttle_pid.reset()
                else:
                    err_norm = np.clip(err_px / float(target_h_px), -1.0, 1.0)
                    throttle_cmd = throttle_pid.step(err_norm)

                throttle_cmd = float(np.clip(throttle_cmd, -max_throttle, max_throttle))
                if 0 < abs(throttle_cmd) < MIN_MOVE_THROTTLE:
                    throttle_cmd = MIN_MOVE_THROTTLE * np.sign(throttle_cmd)
                car.throttle = THROTTLE_SIGN * throttle_cmd

                # Reversing flips the steering->yaw relationship (see markdown above), so
                # only apply the flip while the car is actually driving backward.
                is_reversing = throttle_cmd < 0
                steer_direction = REVERSE_STEERING_SIGN if is_reversing else 1.0
                car.steering = float(np.clip(steer_direction * raw_steering_cmd, -1.0, 1.0))

                status_label.value = (
                    "Status: tracking '%s' | steer=%.2f thr=%.2f err_px=%d%s"
                    % (TARGET_LABEL, car.steering, car.throttle, err_px,
                       " [reversing]" if is_reversing else "")
                )

            overlay = _draw_overlay(frame, target, target_h_px)
            ok, jpeg = cv2.imencode('.jpg', overlay)
            if ok:
                image_widget.value = jpeg.tobytes()

        except Exception:
            status_label.value = "Status: ERROR (see printed traceback) — stopping"
            car.throttle = 0.0
            car.steering = 0.0
            traceback.print_exc()
            running_flag[0] = False
            break

        elapsed = time.time() - loop_start
        time.sleep(max(0.0, period - elapsed))

    # loop exited (Stop pressed or error) -> always leave the car stopped
    car.throttle = 0.0
    car.steering = 0.0
    status_label.value = "Status: stopped"


def on_start_clicked(_):
    if running_flag[0]:
        return
    running_flag[0] = True
    t = threading.Thread(target=_control_loop, daemon=True)
    _loop_thread[0] = t
    t.start()
    status_label.value = "Status: starting..."

def on_stop_clicked(_):
    running_flag[0] = False
    car.throttle = 0.0
    car.steering = 0.0
    status_label.value = "Status: stopped (manual)"

start_button.on_click(on_start_clicked)
stop_button.on_click(on_stop_clicked)

print("Control loop wired up. Use the START/STOP buttons above the sliders.")


## 10. Tuning Tips

- **Car oscillates side to side**: lower `Steer Kp`, or raise `Steer Kd` slightly.
- **Car is sluggish to turn toward the object**: raise `Steer Kp`.
- **Car creeps forward/back even when object is at the right distance**: widen
  `DISTANCE_DEADBAND_PX` in Section 6b, or lower `Dist Kp`.
- **Car surges then overshoots the target distance**: lower `Dist Kp`, add a little more
  damping isn't available on the throttle PID by default (Kd=0.1 already included) — try
  lowering `Dist Ki` first if it overshoots and slowly drifts back.
- **Detector loses the object often / doesn't detect a person easily**: run the Section 4c
  diagnostic cell first to see actual confidence scores. Then, in rough order of impact:
  1. Make sure `USE_COCO_MODEL = True` in Section 4a (COCO is far more robust on people than
     the VOC fallback) and that it actually loaded (`ACTIVE_MODEL == "coco"` printed in 4a/4b).
  2. Lower the "Min conf." slider (or `CONFIDENCE_THRESHOLD` in Section 4b) — 0.25-0.35 is
     often more realistic than the textbook default of 0.5.
  3. Stand back enough that most of your body is in frame — partial/cropped views score lower.
  4. Fix harsh backlighting (e.g. a bright window directly behind you, like in the reference
     photo) — it silhouettes the subject and hurts the detector badly; try facing a light
     source instead of having it behind you.
  5. Confirm `TARGET_LABEL` matches a class the active model actually knows (Section 5).
- **Steering looks correct while approaching but backwards while backing away** (or vice
  versa): flip `REVERSE_STEERING_SIGN` in Section 3 between `-1.0` and `1.0` — see the
  explanation in Section 9 for why reversing needs a different steering sign at all.
- **Everything feels too slow**: this MobileNet-SSD + Jetson Nano combo typically runs
  several FPS on CPU and faster with the CUDA backend — check the printed backend message
  in Section 4b. `CONTROL_LOOP_HZ` in Section 9 won't exceed what the detector can actually
  deliver.


## 11. Emergency Stop / Cleanup

Run this cell any time to force everything to a safe state (also good practice at the end of
a session, or before closing the notebook / powering off).


In [ ]:
# ---------------------------------------------------------------------------
# 11. Emergency stop & cleanup
# ---------------------------------------------------------------------------
running_flag[0] = False
time.sleep(0.2)  # give the loop thread a moment to exit cleanly

car.throttle = 0.0
car.steering = 0.0

try:
    camera.running = False
except Exception:
    pass

status_label.value = "Status: stopped (cleanup)"
print("Car stopped, camera released. Safe to close the notebook.")
